# 03 - Training and Evaluation

This notebook initializes NRMS, trains it on negative-sampled MIND-small impressions, plots the loss curve, and evaluates ranking quality on the dev split.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.optim import Adam

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.append(str(SRC_DIR))

from data_loader import (
    build_vocabulary,
    combine_news,
    create_eval_dataloader,
    create_train_dataloader,
    encode_news_titles,
    load_glove_embeddings,
    load_news,
    set_seed,
)
from evaluate import evaluate_model
from model import NRMSModel

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Load Processed Inputs

For clarity, this notebook rebuilds preprocessing in-memory. For repeated experiments, load the artifacts saved by `02_preprocessing.ipynb`.

In [2]:
TRAIN_DIR = PROJECT_ROOT / 'data' / 'MINDsmall_train'
DEV_DIR = PROJECT_ROOT / 'data' / 'MINDsmall_dev'
GLOVE_PATH = PROJECT_ROOT / 'data' / 'glove' / 'glove.6B.300d.txt'

train_news = load_news(TRAIN_DIR / 'news.tsv')
dev_news = load_news(DEV_DIR / 'news.tsv')
vocab = build_vocabulary(train_news, min_frequency=2)
news_title_map = encode_news_titles(combine_news(train_news, dev_news), vocab, max_length=30)
embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab, embedding_dim=300, seed=42)

train_loader = create_train_dataloader(
    TRAIN_DIR / 'behaviors.tsv', news_title_map, batch_size=64, history_size=50, negative_sampling_ratio=4, seed=42
)
dev_loader = create_eval_dataloader(DEV_DIR / 'behaviors.tsv', news_title_map, history_size=50)
len(train_loader.dataset), len(dev_loader.dataset)

Loaded GloVe vectors for 19,051/20,881 tokens (91.2%).


(236344, 73152)

## Initialize NRMS

The model uses multi-head self-attention in both news and user encoders, then scores candidates with a dot product.

In [3]:
model = NRMSModel(
    embedding_matrix=embedding_matrix,
    num_heads=15,
    attention_dim=200,
    dropout=0.2,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)
sum(parameter.numel() for parameter in model.parameters())

7107500

## Train Model

This compact loop is notebook-friendly. For full experiments with checkpointing, use `python src/train.py --epochs 5`.

In [4]:
epochs = 3
loss_curve = []

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for history, candidates, labels in train_loader:
        history = history.to(device)
        candidates = candidates.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(history, candidates)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)
    epoch_loss = total_loss / total_examples
    loss_curve.append(epoch_loss)
    print(f'Epoch {epoch}: loss={epoch_loss:.4f}')

Epoch 1: loss=1.4249


KeyboardInterrupt: 

## Plot Loss Curve

A steadily decreasing loss indicates that the model is learning to separate clicked items from sampled negatives.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), loss_curve, marker='o')
plt.title('NRMS Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.tight_layout()
plt.show()

## Evaluate

Evaluation preserves full impression groups and reports AUC, MRR, nDCG@5, and nDCG@10.

In [ ]:
metrics = evaluate_model(model, dev_loader, device)
metrics

In [ ]:
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), MODELS_DIR / 'nrms_notebook_state_dict.pt')
print(f'Saved notebook model weights to {MODELS_DIR / "nrms_notebook_state_dict.pt"}')